# Session 2 · Part 1 — CUNQA Foundations · solutions

**15:00 – 15:45**

---

Every exercise of `04-cunqa-foundations.ipynb` completed. Same cells, same
order, same checks — only the `TODO`s are filled in.

---

Notebooks 01–03 were about the *programming model*. This one is about the
machine underneath it: what a virtual QPU actually is, what a run of one costs,
and what NetQMPI decides on your behalf every time you write `qsend`.

The argument is made by taking something away. In notebook 03 you teleported a
qubit in six lines. Here you write **the same program against CUNQA directly**,
with no NetQMPI in it, and count what became yours to choose. Then you write the
NetQMPI version again and run both.

| Notebook | Topic | Time |
|---|---|---|
| **04** (this one) | CUNQA: vQPUs, families, and the raw API | 15:00–15:45 |
| 05 | Distributed inverse QFT and phase estimation | 15:45–16:15 |

> **The vQPU family raised in section 2 stays up for both notebooks.** One
> allocation serves the whole afternoon. Do not run `qdrop` until the end of
> notebook 05.

## 0 · One-time setup

Two things the kernel does not do for you.

`qraise` and `qdrop` are on `PATH` already, from `$HOME/bin`. The `netqmpi`
launcher is **not**: the kernel starts by exec'ing the venv's interpreter
directly and never adds the venv's `bin/` to the environment, so `!netqmpi` in a
cell would not be found. One line fixes it, and then every launch in these two
notebooks looks exactly like the `!mpirun` and `!netqmpi` lines of notebooks
01–03.

CUNQA itself lives in `$HOME`, which is why every CUNQA import in this stack is
preceded by a `sys.path` line.

In [ ]:
import json
import os
import sys
from pathlib import Path

# The venv's bin/, so that `!netqmpi ...` resolves in a shell cell.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

# CUNQA lives in $HOME — the same line every NetQMPI CUNQA adapter starts with.
sys.path.append(os.getenv("HOME"))

from cunqa.circuit import CunqaCircuit
from cunqa.qc_protocols import qsend, qrecv
from cunqa.qjob import gather
from cunqa.qpu import get_QPUs, qdrop, qraise, run

WORK = Path("notebook_apps")
WORK.mkdir(exist_ok=True)

FAMILY = "netqmpi_notebook"   # the vQPU family this session owns
SIZE = 3                      # nodes / ranks / vQPUs
SHOTS = 1024

print("python :", sys.executable)
print("apps in", WORK.resolve())

In [ ]:
# EXAMPLE — all three launchers must resolve before anything else works.
!which netqmpi qraise qdrop

---
## 1 · What CUNQA is, and why to use it

Notebook 03 used CUNQA as a backend flag. Here is what is behind the flag.

CUNQA emulates a QPU as a **bundle of classical HPC resources plus a simulation
backend** — a *virtual QPU*, a vQPU. Because a vQPU is a classical resource on a
cluster, its whole lifecycle is expressed in the scheduler's terms: `qraise`
submits a SLURM job, the vQPUs come up as services inside it, and `qdrop`
releases them. The Slurm installation from notebook 01 was never scaffolding for
the classical half; it is the mechanism the quantum half runs on.

**Why bother emulating, when a statevector simulator exists?** Because the thing
being studied is not the circuit. It is everything around it:

| A plain simulator gives you | A vQPU family gives you |
|---|---|
| One process, one register | *N* addressable endpoints, each with its own queue |
| Gates | Gates **plus** entanglement requests between endpoints, and the classical messages that go with them |
| No topology | Co-located or spread across nodes, with the interconnect in between |
| No scheduler | A real allocation, with a real cost and a real wall clock |

That is the point of the tool: a distributed quantum program has failure modes
that only exist *between* the QPUs, and you cannot see them on a machine that has
only one.

### Families

Each `qraise` deploys a group of vQPUs that share one configuration and one
executor. That group is a **family**, and it is the unit that matters:

- Every vQPU of a family is raised from the **same definition** — same qubit
  counts, same simulator, same noise.
- The family's executor starts a round only when **every** vQPU in it has
  submitted something. A vQPU left out does not idle; it stalls the round. You
  will meet this in exercise 6, and NetQMPI handles it for you by keeping the
  spare vQPUs of an oversized family busy with a trivial circuit.
- `quantum_comm=True` is what makes the vQPUs able to entangle with each other.
  Without it there is no `gen_ent`, and every protocol in this notebook fails.
- A run cannot spread across families. With exactly one family up, `netqmpi
  --cunqa` attaches to it with no configuration at all — which is why every
  launch below is a bare command line.

### The one number that decides whether your run finishes

The executor simulates **the whole family in one register**, spanning every qubit
each vQPU *declares* — whether the circuits touch it or not. So the cost of a run
is set by

    (data qubits + comm qubits) × number of vQPUs

and *not* by your circuit. A vQPU definition is a JSON file whose `num_qubits` is
the pair `[data, comm]`: data qubits are yours, comm qubits are the ones the
teledata and telegate protocols consume.

> ### ⚠ The one setting that will ruin your afternoon
>
> With a statevector simulator that single register is 2^N amplitudes, at 16
> bytes each:
>
> | vQPU definition | 2 vQPUs | 3 vQPUs | 4 vQPUs |
> |---|---|---|---|
> | `[2, 2]` (today) | 4 KiB | 64 KiB | 1 MiB |
> | `[4, 4]` | 1 MiB | 256 MiB | **64 GiB** |
> | `[8, 8]` | **64 GiB** | — | — |
>
> If a cell hangs, this is the first thing to check — and the number to check is
> in the vQPU definition file, never in your circuit.

---
## 2 · Raising the family for this afternoon

Three vQPUs — one per rank — sized for every exercise in notebooks 04 and 05 at
once, so that one allocation serves both and no run pays a SLURM cost.

`[2, 2]` is not a guess. **Two data qubits** because exercise 9's rank 0 holds a
counting qubit *and* the eigenstate; every other rank uses one and wastes the
second, because a family is raised from a single definition. **Two comm qubits**
because the inverse QFT of exercise 8 keeps two `expose` windows open at once,
and a window costs one comm qubit on every rank inside it.

Four qubits times three vQPUs is a twelve-qubit register: 4096 amplitudes, 64
KiB. Read that off the definition, not off the circuits.

In [ ]:
VQPU = WORK / "notebook_vqpu.json"
VQPU.write_text(json.dumps({
    "name": "netqmpi_notebook",
    "description": (
        "vQPU for the exercises of session 2. 2 data qubits (exercise 9's rank 0 "
        "holds a counting qubit and the eigenstate) and 2 comm qubits (the two "
        "overlapping expose windows of the inverse QFT). 4 qubits x 3 vQPUs = 12 "
        "qubits in the family's single simulated register."
    ),
    "num_qubits": [2, 2],
}, indent=4))

print(VQPU.read_text())

In [ ]:
qpus = get_QPUs(co_located=True, family=FAMILY) or []

if len(qpus) < SIZE:
    # Blocks until SLURM reports the job running and every vQPU has registered.
    qraise(SIZE, "02:00:00",
           simulator="Munich",
           co_located=True,
           quantum_comm=True,          # the vQPUs must be able to entangle
           backend=str(VQPU),
           family=FAMILY)
    qpus = get_QPUs(co_located=True, family=FAMILY) or []

for qpu in qpus:
    print(f"{qpu.id}  family={qpu.family}  qubits={qpu.backend.get('num_qubits')}")
assert len(qpus) >= SIZE, f"need {SIZE} vQPUs, found {len(qpus)}"

In [ ]:
# EXAMPLE — they are scheduler jobs, like everything else on this machine.
!squeue

---
## Exercise 6 — communicating two vQPUs

This is all of `examples/netqmpi/1_send_recv.py`, the program you ran in notebook
03. Rank 0 prepares `|+>` and hands it to rank 1, which measures it:

```python
with comm:
    circuit = env.create_circuit(num_qubits=1, num_clbits=1)

    if rank == 0:
        circuit.h(0)                            # prepare |+>
        comm.qsend(circuit, [0], next_rank)     # and give it away
    else:
        comm.qrecv(circuit, [0], previous_rank)
        circuit.measure(0, 0)
```

**Write it again without NetQMPI**: two `CunqaCircuit`s built by hand,
teleportation with `cunqa.qc_protocols.qsend`/`qrecv`, submitted with `run`.

The point is not that it is hard. It is that four things NetQMPI *derived* are
now yours to choose, and each one is a way to get it wrong:

| | NetQMPI | you |
|---|---|---|
| circuit id | `f"rank_{rank}"` | you name them — and the names are the addresses |
| comm qubits | borrowed from a pool per protocol block | declared up front in `(data, comm)` |
| protocol clbits | own `netqmpi_protocol` register | yours to allocate, apart from your own bits |
| pairing the two halves | tag derived from the ranks | the same string on both sides, or the run hangs |

What you need:

```python
CunqaCircuit((num_data, num_comm), num_clbits, id="...")
circuit.add_cl_register("name", n)  -> name    # circuit.classical_regs[name] -> [bits]
circuit.comm_qubits                 -> [indices]

qsend(circuit, data_qubit, comm_qubit, clbits, recving_circuit="<id>", tag="...")
qrecv(circuit, data_qubit, comm_qubit, clbits, control_circuit="<id>", tag="...")
```

Both need **one** comm qubit and **two** classical bits.

In [ ]:
def build_send_recv():
    """
    Teleport |+> from rank 0 to rank 1, in plain CUNQA.

    Returns:
        The two circuits, in rank order.
    """
    # Both halves of a transfer are paired at run time by this string. NetQMPI
    # builds it from the ranks involved plus a counter — "teledata_<src>_<dst>_<n>" —
    # so that both sides name the transfer identically without ever talking.
    TAG = "teledata_0_1_0"

    # ---- rank 0: one data qubit, one comm qubit, one clbit of its own ----
    c0 = CunqaCircuit((1, 1), 1, id="rank_0")
    proto0 = c0.add_cl_register("netqmpi_protocol", 2)

    c0.h(0)                                    # prepare |+> on the data qubit
    qsend(
        c0,
        0,                                     # the data qubit being given away
        c0.comm_qubits[0],                     # its half of the Bell pair
        c0.classical_regs[proto0],             # the two Bell-measurement outcomes
        recving_circuit="rank_1",              # the id is the address
        tag=TAG,
    )

    # ---- rank 1: the same resources, on the receiving side ----
    c1 = CunqaCircuit((1, 1), 1, id="rank_1")
    proto1 = c1.add_cl_register("netqmpi_protocol", 2)

    qrecv(
        c1,
        0,                                     # where the state lands
        c1.comm_qubits[0],                     # its half of the Bell pair
        c1.classical_regs[proto1],             # the two bits it receives
        control_circuit="rank_0",
        tag=TAG,
    )

    c1.measure(0, 0)                           # into the user's own clbit

    return [c0, c1]

<details>
<summary><b>Hints</b></summary>

* On a `(1, 1)` circuit, `comm_qubits` is `[1]` — comm qubits are numbered after
  the data ones.
* `add_cl_register` returns the name it used; `classical_regs[proto0]` is
  `[1, 2]`, straight into `qsend`/`qrecv`.
* `recving_circuit` / `control_circuit` take the other circuit's **id string**.
* The tag must match byte for byte. Get it wrong and nothing errors — each vQPU
  waits forever for a partner that never names it.

</details>

In [ ]:
circuits = build_send_recv()

# The family has three vQPUs and this program uses two. The spare one cannot
# just be left out: CUNQA runs one executor per family and starts a round only
# once EVERY vQPU has submitted something, so an idle one stalls the round
# rather than sitting it out. Give it a throwaway circuit.
spare = CunqaCircuit((1, 0), 1, id="spare")
spare.measure(0, 0)

results = gather(run(circuits + [spare], qpus, shots=SHOTS))

for result in results:
    if result.id != "spare":
        print(f"{result.id}: {result.counts}")

**Read the counts before moving on.** `rank_1` splits roughly 50/50, which is
what `|+>` looks like — and also what a *broken* channel returning noise looks
like. Change `c0.h(0)` to `c0.x(0)` and run it again: rank 1 must then report `1`
on essentially every shot. That is the test that actually distinguishes the two.

`rank_0` contributes nothing either way. Teleportation *moves* a state, so by the
time the protocol is over rank 0's data qubit is back in `|0>` and its only
measurements are the protocol's own, taken with `save=False`.

Now look at what you built. `qsend` and `qrecv` are not primitives: they are a
handful of instructions around `gen_ent`, the request for a Bell pair between two
vQPUs.

In [ ]:
# EXAMPLE — the instructions your two calls expanded into
for circuit in circuits:
    print(f"== {circuit.id}: {circuit.num_qubits[0]} data + {circuit.num_qubits[1]} comm "
          f"qubits, registers {circuit.classical_regs}")
    for instruction in circuit.instructions:
        print("   ", instruction)
    print()

---
## Exercise 7 — the same program, the other programming model

Now write it in NetQMPI and run it against the very same three vQPUs.

This is two lines of work — you wrote them in notebook 03 — and the exercise is
what comes after: **compare the two files.** Same protocol, same counts, same
family, and everything in the table of exercise 6 derived rather than chosen.

Same result-printing convention as notebook 03: every rank's circuit is
submitted together, when the last rank leaves the `with comm:` block, so **only
that rank sees `comm.results` — and it sees all of them.** Hence the
`if comm.results:` guard at the bottom of every program.

In [ ]:
%%writefile notebook_apps/ex7_send_recv.py
"""
Exercise 7: the program of exercise 6, written in NetQMPI.

Same teleportation, same two circuits underneath — but the circuit ids, the comm
qubit, the protocol register and the pairing tag are all derived instead of
chosen.

Run with::

    netqmpi -n 2 notebook_apps/ex7_send_recv.py --cunqa --shots 1024
"""
from netqmpi.sdk.environment import Environment


def main(env: Environment = None):
    comm = env.comm
    rank = comm.rank

    next_rank = comm.get_next_rank(rank)
    previous_rank = comm.get_prev_rank(rank)

    with comm:
        circuit = env.create_circuit(num_qubits=1, num_clbits=1)

        if rank == 0:
            circuit.h(0)                       # prepare |+>
            comm.qsend(circuit, [0], next_rank)
        else:
            comm.qrecv(circuit, [0], previous_rank)
            circuit.measure(0, 0)

    # Only the last rank out of the block holds the results, and it holds
    # everyone's.
    if comm.results:
        for other, counts in comm.results.items():
            print(f"rank_{other}: {counts}")

<details>
<summary><b>Hints</b></summary>

* `comm.qsend(circuit, [0], next_rank)` and
  `comm.qrecv(circuit, [0], previous_rank)` — the qubit argument is a *list* of
  indices, because the same call moves several at once.
* No tag, no comm qubit, no protocol register: all three are derived from the
  ranks and the call order.

</details>

In [ ]:
!netqmpi -n 2 notebook_apps/ex7_send_recv.py --cunqa --shots 1024

**No `--config`, and no family named on the command line.** With exactly one
family up, that is all the adapter needs; it attaches to what is running and
leaves the allocation alone afterwards, which is why every launch in notebook 05
is this cheap.

**Two ranks on a three-vQPU family.** The spare vQPU got a throwaway circuit from
the adapter — the same thing you did by hand in exercise 6, except that you had
to know to do it. An oversized family is wasteful, never wrong.

Now put the two files side by side. Roughly thirty lines against six, and the
difference is entirely the four rows of the exercise-6 table: the tag, the
comm-qubit index, the protocol register and the circuit ids. None of those has a
*correct* value you could look up — they only have to **agree between the two
halves**, and nothing checks that they do. That is the class of bug NetQMPI
removes: not a hard one, a silent one.

---
## Recap

1. **A vQPU is a scheduler allocation.** `qraise` submits a job, `qdrop` releases
   it, and everything in between attaches to a **family** by name.
2. **The family is simulated as one register.** Cost is
   `(data + comm) × vQPUs`, read off the *definition*, not off your circuit.
3. **`qsend` is not a primitive.** You saw the instructions: a `gen_ent` request,
   a local Bell measurement, two classical bits, two conditional corrections.
4. **Both programming models drive the same protocol.** What NetQMPI buys is not
   capability, it is the four decisions of exercise 6 — the ones that fail
   silently when they drift apart.

Leave the family up. **On to notebook 05.**